# FlagOS Challenge: `logaddexp` Operator
This notebook provides the implementation for `torch.logaddexp` using Triton and the FlagGems `@pointwise_dynamic` decorator.

In [ ]:
!pip install -q flag_gems triton torch

In [ ]:
import torch
import triton
import triton.language as tl
import time

print(f"PyTorch: {torch.__version__}")
print(f"Triton: {triton.__version__}")

## 1. FlagGems Implementation
Mathematically: `log(exp(x) + exp(y))`.
To prevent overflowing when `x` or `y` are large, we use the numerically stable branchless formulation:
`max(x, y) + log(1 + exp(-abs(x - y)))`

In [ ]:
import logging
from flag_gems.utils import pointwise_dynamic

logger = logging.getLogger(__name__)

@pointwise_dynamic(promotion_methods=[(0, "INT_TO_FLOAT"), (1, "INT_TO_FLOAT")])
@triton.jit
def logaddexp_func(x, y):
    x_f32 = x.to(tl.float32)
    y_f32 = y.to(tl.float32)
    
    # Safely handle +/- infinity equality to prevent NaN from diff (inf - inf)
    diff = tl.where(x_f32 == y_f32, 0.0, tl.abs(x_f32 - y_f32))
    max_val = tl.maximum(x_f32, y_f32)
    
    # Provide strict NaN propagation required by PyTorch
    has_nan = (x_f32 != x_f32) | (y_f32 != y_f32)
    max_val = tl.where(has_nan, float('nan'), max_val)
    
    # Numerically stable calculation
    return max_val + tl.log(1.0 + tl.exp(-diff))

def logaddexp(x, y):
    logger.debug("GEMS LOGADDEXP")
    return logaddexp_func(x, y)

def logaddexp_(x, y):
    logger.debug("GEMS LOGADDEXP_")
    return logaddexp_func(x, y, out0=x)


## 2. Standalone Triton Kernels (For Local Colab Testing)

In [ ]:
@triton.autotune(
    configs=[
        triton.Config({'BLOCK_SIZE': 256}),
        triton.Config({'BLOCK_SIZE': 512}),
        triton.Config({'BLOCK_SIZE': 1024}),
        triton.Config({'BLOCK_SIZE': 2048}),
    ],
    key=['n_elements'],
)
@triton.jit
def logaddexp_kernel(x_ptr, y_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    
    x_f32 = x.to(tl.float32)
    y_f32 = y.to(tl.float32)
    
    diff = tl.where(x_f32 == y_f32, 0.0, tl.abs(x_f32 - y_f32))
    max_val = tl.maximum(x_f32, y_f32)
    
    has_nan = (x_f32 != x_f32) | (y_f32 != y_f32)
    max_val = tl.where(has_nan, float('nan'), max_val)
    
    output = max_val + tl.log(1.0 + tl.exp(-diff))
    tl.store(output_ptr + offsets, output, mask=mask)

def triton_logaddexp(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    assert x.shape == y.shape, "Shapes must match for basic prototyping (FlagGems dynamically handles broadcasting)"
    output = torch.empty_like(x)
    n_elements = x.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
    logaddexp_kernel[grid](x, y, output, n_elements)
    return output

## 3. Correctness & Accuracy Tests

In [ ]:
def test_correctness():
    print("=" * 30, "CORRECTNESS TESTS", "=" * 30)
    
    # Standard Values
    x1 = torch.tensor([1.0, -2.0, 0.0, 10.0, 50.0], device='cuda')
    y1 = torch.tensor([2.0, -1.0, 0.0, 50.0, 10.0], device='cuda')
    res1 = triton_logaddexp(x1, y1)
    exp1 = torch.logaddexp(x1, y1)
    
    match_base = torch.allclose(res1, exp1, atol=1e-5)
    print(f"Standard Float Check:     {'PASS' if match_base else 'FAIL'}")
    
    # Huge numerical stability check (exp(100) and exp(99) would crash naïve PyTorch)
    x2 = torch.tensor([1000.0, -1000.0, 800.0], device='cuda')
    y2 = torch.tensor([1000.0, 1000.0, -800.0], device='cuda')
    res2 = triton_logaddexp(x2, y2)
    exp2 = torch.logaddexp(x2, y2)
    match_stab = torch.allclose(res2, exp2, atol=1e-5)
    print(f"Numeric Stability Check:  {'PASS' if match_stab else 'FAIL'}")
    
    # Infinity & NaN Extreme Edge Cases
    x_edge = torch.tensor([float('inf'), float('-inf'), float('inf'), float('nan')], device='cuda')
    y_edge = torch.tensor([float('inf'), float('-inf'), float('-inf'), 5.0], device='cuda')
    
    res_edge = triton_logaddexp(x_edge, y_edge)
    exp_edge = torch.logaddexp(x_edge, y_edge)
    
    # Nans can't be compared with ==, so use torch.isnan()
    match_inf_inf = res_edge[0] == exp_edge[0]
    match_ninf_ninf = res_edge[1] == exp_edge[1]
    match_inf_ninf = res_edge[2] == exp_edge[2]
    match_nan = torch.isnan(res_edge[3]) and torch.isnan(exp_edge[3])
    
    match_edge = match_inf_inf and match_ninf_ninf and match_inf_ninf and match_nan
    print(f"Extreme Infinity Check:   {'PASS' if match_edge else 'FAIL'}")

test_correctness()

## 4. Benchmark Performance

In [ ]:
def benchmark():
    print("\n" + "=" * 25, "PERFORMANCE", "=" * 25)
    print(f"{'Size':>12} {'Op':>10} {'Triton(us)':>12} {'PyTorch(us)':>12} {'Speedup':>8}")
    print("-" * 65)
    
    sizes = [1048576, 16777216]
    
    def run_bench(fn, x, y):
        n_warmup, n_iter = 50, 200
        for _ in range(n_warmup): fn(x, y)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_iter): fn(x, y)
        torch.cuda.synchronize()
        return (time.perf_counter() - t0) / n_iter * 1e6

    for size in sizes:
        x_tensor = torch.rand(size, device='cuda') * 10.0
        y_tensor = torch.rand(size, device='cuda') * 10.0
        
        t_tri = run_bench(triton_logaddexp, x_tensor, y_tensor)
        t_py = run_bench(torch.logaddexp, x_tensor, y_tensor)
        
        sdp = t_py / t_tri
        print(f"{size:>12} {'logaddexp':>10} {t_tri:>12.1f} {t_py:>12.1f} {sdp:>7.2f}x")

benchmark()